# Table summary


In [ ]:
# general
from pathlib import Path
import pandas as pd
import geopandas as gpd
import xarray as xr
import yaml 
import os

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes/')
os.chdir(wd)

# import config and local module
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

scenarios       = [s["id"] for s in cfg["scenarios"]]
scenarios_names = [s["name"] for s in cfg["scenarios"]]
colors          = [s["color"] for s in cfg["scenarios"]]

## Data

In [ ]:
# polygons / shapes
df = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index("gauge_id")
df = df[['glacier_area_RGI60', 'basin_area', 'basin_cenlat']]
df["glacier_area_RGI60"] = df.glacier_area_RGI60 * 0.01 * df.basin_area

# drought metrics
droughts_metrics = pd.read_csv("data/hydro/droughts_hydroatlas_whatif.csv")
droughts_metrics = droughts_metrics.pivot(index="gauge_id", columns="scenario")
droughts_metrics.columns = ['_'.join(col).strip() for col in droughts_metrics.columns.values]
droughts_metrics.index = droughts_metrics.index.astype(str)
df = df.join(droughts_metrics)

# drought extent
droughts_extent = (pd.read_csv("data/hydro/droughts_hydroatlas_area_ts.csv", index_col=["scenario", "date"], parse_dates=["date"])
                   .rolling(window=90, min_periods=1, center=True).mean()
                   .groupby(level='scenario').mean())
droughts_extent_reg = pd.DataFrame({
    region: droughts_extent[[c for c in droughts_extent.columns if lat_min <= int(c) < lat_max]].sum(axis=1)
    for region, (lat_max, lat_min) in cfg["regions"].items()
}).T
droughts_extent_reg = (100 * (1 - droughts_extent_reg[['15C', '28C', '40C']].div(droughts_extent_reg['historical'], axis=0))).T
droughts_extent_reg.index = droughts_extent_reg.index.map(lambda x: f"drought_extent_{x}")

# total runoff in mm/day to m3/s
hydroatlas_runoff_sim = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_hist.nc").q_mm_sim
hydroatlas_runoff_sim = hydroatlas_runoff_sim.reindex(gauge_id=df.index)
hydroatlas_runoff_sim = hydroatlas_runoff_sim*(df.basin_area.values*1e6)[:, None] / (1000*86400)
hydroatlas_runoff_sim = hydroatlas_runoff_sim.resample(date="YS").mean().mean(dim = "date")
df = df.join(hydroatlas_runoff_sim.rename("total_runoff").to_pandas())

# glacier runoff in m3/s
hydroatlas_gmelt = xr.open_dataset("data/glacier/run_output_historical.nc").glacier_melt
hydroatlas_gmelt = hydroatlas_gmelt.reindex(gauge_id=df.index, fill_value=0)
hydroatlas_gmelt = hydroatlas_gmelt.resample(date="YS").mean().mean(dim = "date")
df = df.join(hydroatlas_gmelt.rename("glacier_melt").to_pandas())

# glacier evolution
df_glaciers = pd.read_csv("data/glacier/glacier_metadata_evolution.csv", index_col=0)
df_glaciers["region_name"] = pd.cut(df_glaciers["cenlat"], 
                                    bins=sorted({v for b in cfg["regions"].values() for v in b}), 
                                    labels=list(cfg["regions"].keys()))
df_glaciers = df_glaciers.groupby("region_name", observed=False).sum(numeric_only=True)
df_glaciers = df_glaciers[['volume_2100_15C', 'volume_2100_28C', 'volume_2100_40C']].div(df_glaciers['volume_2020_historical'], axis=0)
df_glaciers = (100 * (1 - df_glaciers)).transpose()

# water stress
wsi = pd.read_parquet("data/hydro/water_stress.parquet")
wsi = wsi.set_index(["scenario", "month", "gauge_id"])
wsi = wsi[["WSI"]].loc[wsi.index.get_level_values("month").isin([1, 2, 3])]
wsi = wsi.groupby(["scenario", "gauge_id"]).mean()
wsi = wsi.unstack("scenario").droplevel(0, axis=1)
wsi.columns = "WSI_" + wsi.columns
wsi_historical = wsi["WSI_historical"]
wsi = wsi.div(wsi_historical, axis=0).sub(1).mul(100)
df = df.join(wsi)

## Table

In [ ]:
# this create a dataframe with the same columns as the number of regions
df_agg = pd.DataFrame(columns=cfg["regions"].keys())

# these columns are summed, the rest are averaged (weighted by basin area)
sum_cols = ['glacier_area_RGI60', 'basin_area', 'total_runoff', 'glacier_melt']

for region, (lat_max, lat_min) in cfg["regions"].items():

    mask = (df['basin_cenlat'] >= lat_min) & (df['basin_cenlat'] < lat_max)

    # weithed mean
    mean_cols = df.columns.difference(sum_cols)
    weights = df.loc[mask, 'basin_area']
    weighted_means = df.loc[mask, mean_cols].apply(lambda x: (x * weights).sum() / weights[x.notna()].sum())

    # normal sum
    df_agg[region] = pd.concat([
        df.loc[mask, sum_cols].sum(),
        weighted_means
    ])

df_agg.loc["glacier_area_rel"] = df_agg.loc["glacier_area_RGI60"] * 100 / df_agg.loc["basin_area"]
df_agg.loc["glacier_melt_rel"] =  df_agg.loc["glacier_melt"] * 100 / df_agg.loc["total_runoff"]
df_agg = pd.concat([df_agg, df_glaciers], axis=0)
df_agg = pd.concat([df_agg, droughts_extent_reg], axis=0)
df_agg = df_agg.round(1)
df_agg.to_csv("figures/table_summary.csv")